# Gaussian Naive Bayes Implementation - SOLUTION

## Complete working solution with line-by-line implementation

This notebook contains the complete solution for implementing Gaussian Naive Bayes from scratch.

In [ ]:
# Step 1: Import Libraries and Load Data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import GaussianNB  # For comparison
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('default')
%matplotlib inline

print("Libraries imported successfully!")

In [ ]:
# Step 2: Load and Explore the Dataset
iris = load_iris()
X = iris.data  # Features: sepal length, sepal width, petal length, petal width
y = iris.target  # Target: 0=setosa, 1=versicolor, 2=virginica

feature_names = iris.feature_names
class_names = iris.target_names

# Create DataFrame for exploration
df = pd.DataFrame(X, columns=feature_names)
df['species'] = y
df['species_name'] = df['species'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

# Display dataset information
print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {len(class_names)}")
print(f"Class names: {list(class_names)}")
print(f"Feature names: {list(feature_names)}")
print("\nFirst 5 rows:")
print(df.head())

# Check class distribution
print("\nClass distribution:")
print(df['species_name'].value_counts())

In [ ]:
# Step 3: Visualize the Data
# Create histograms for each feature
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, feature in enumerate(feature_names):
    for species_idx, species in enumerate(class_names):
        data = df[df['species'] == species_idx][feature]
        axes[i].hist(data, alpha=0.6, label=species, bins=15)
    
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Frequency')
    axes[i].set_title(f'Distribution of {feature}')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate basic statistics by class
print("Basic statistics by species:")
print(df.groupby('species_name')[feature_names].describe().round(3))

In [ ]:
# Step 4: Split the Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% for testing
    random_state=42,    # For reproducibility
    stratify=y          # Maintain class proportions
)

# Display the shapes
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

# Check class distributions
unique_train, counts_train = np.unique(y_train, return_counts=True)
unique_test, counts_test = np.unique(y_test, return_counts=True)

print("\nTraining set class distribution:")
for i, count in zip(unique_train, counts_train):
    print(f"  {class_names[i]}: {count} samples")

print("\nTesting set class distribution:")
for i, count in zip(unique_test, counts_test):
    print(f"  {class_names[i]}: {count} samples")

In [ ]:
# Step 5: Implement Gaussian Naive Bayes from Scratch
class GaussianNaiveBayesFromScratch:
    """
    Gaussian Naive Bayes classifier implemented from scratch.
    
    This implementation assumes that continuous features follow a Gaussian (normal) 
    distribution and that features are conditionally independent given the class.
    """
    
    def __init__(self):
        """
        Initialize the Gaussian Naive Bayes classifier.
        """
        self.classes_ = None
        self.class_priors_ = None  # P(C_k)
        self.feature_means_ = None  # μ_ik
        self.feature_vars_ = None   # σ²_ik
    
    def fit(self, X, y):
        """
        Train the Gaussian Naive Bayes classifier.
        
        Parameters:
        X : array-like, shape (n_samples, n_features)
            Training data
        y : array-like, shape (n_samples,)
            Target labels
        """
        # Convert inputs to numpy arrays for consistent handling
        X = np.array(X)
        y = np.array(y)
        
        # Get the number of samples and features
        n_samples, n_features = X.shape
        
        # Find unique classes and sort them
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        
        # Initialize arrays to store parameters
        self.class_priors_ = np.zeros(n_classes)  # P(C_k)
        self.feature_means_ = np.zeros((n_classes, n_features))  # μ_ik 
        self.feature_vars_ = np.zeros((n_classes, n_features))   # σ²_ik
        
        # Loop through each class to calculate parameters
        for idx, class_label in enumerate(self.classes_):
            
            # Get samples belonging to this class
            class_mask = (y == class_label)
            X_class = X[class_mask]  # All samples of this class
            
            # Calculate class prior P(C_k)
            self.class_priors_[idx] = X_class.shape[0] / n_samples
            
            # Calculate mean for each feature in this class
            self.feature_means_[idx, :] = np.mean(X_class, axis=0)
            
            # Calculate variance for each feature in this class
            self.feature_vars_[idx, :] = np.var(X_class, axis=0)
            
            # Add small epsilon to prevent division by zero
            self.feature_vars_[idx, :] += 1e-9  # Numerical stability
        
        # Return self for method chaining
        return self
    
    def _gaussian_pdf(self, x, mean, variance):
        """
        Calculate Gaussian probability density function.
        
        Parameters:
        x : float or array-like
            Value(s) to calculate probability for
        mean : float
            Mean (μ) of the Gaussian distribution
        variance : float
            Variance (σ²) of the Gaussian distribution
            
        Returns:
        probability : float or array-like
            Probability density value(s)
        """
        
        # Calculate the normalization constant (1/√(2πσ²))
        normalization = 1.0 / np.sqrt(2 * np.pi * variance)
        
        # Calculate the exponent (-(x-μ)²/(2σ²))
        exponent = -0.5 * ((x - mean) ** 2) / variance
        
        # Calculate the full probability density
        probability = normalization * np.exp(exponent)
        
        return probability
    
    def predict_proba(self, X):
        """
        Predict class probabilities for input samples.
        
        Parameters:
        X : array-like, shape (n_samples, n_features)
            Samples to predict probabilities for
            
        Returns:
        probabilities : array, shape (n_samples, n_classes)
            Predicted class probabilities for each sample
        """
        
        # Convert input to numpy array
        X = np.array(X)
        n_samples, n_features = X.shape
        n_classes = len(self.classes_)
        
        # Initialize array to store log probabilities
        # We use log probabilities to avoid numerical underflow
        log_probabilities = np.zeros((n_samples, n_classes))
        
        # Loop through each class
        for class_idx, class_label in enumerate(self.classes_):
            
            # Start with log of class prior P(C_k)
            log_prior = np.log(self.class_priors_[class_idx])
            
            # Calculate log likelihood for each feature
            log_likelihood = 0
            
            for feature_idx in range(n_features):
                
                # Get mean and variance for this feature and class
                mean = self.feature_means_[class_idx, feature_idx]
                variance = self.feature_vars_[class_idx, feature_idx]
                
                # Calculate log of Gaussian PDF for this feature
                # log(PDF) = log(1/√(2πσ²)) - (x-μ)²/(2σ²)
                log_normalization = -0.5 * np.log(2 * np.pi * variance)
                log_exponent = -0.5 * ((X[:, feature_idx] - mean) ** 2) / variance
                log_pdf = log_normalization + log_exponent
                
                # Add to total log likelihood (multiplication becomes addition in log space)
                log_likelihood += log_pdf
            
            # Combine prior and likelihood
            log_probabilities[:, class_idx] = log_prior + log_likelihood
        
        # Convert back to probabilities and normalize
        # Subtract max for numerical stability before exponentiating
        log_probabilities -= np.max(log_probabilities, axis=1, keepdims=True)
        probabilities = np.exp(log_probabilities)
        
        # Normalize so probabilities sum to 1
        probabilities /= np.sum(probabilities, axis=1, keepdims=True)
        
        return probabilities
    
    def predict(self, X):
        """
        Predict class labels for input samples.
        
        Parameters:
        X : array-like, shape (n_samples, n_features)
            Samples to predict
            
        Returns:
        predictions : array, shape (n_samples,)
            Predicted class labels
        """
        
        # Get probabilities for all classes
        probabilities = self.predict_proba(X)
        
        # Find the class index with maximum probability
        predicted_class_indices = np.argmax(probabilities, axis=1)
        
        # Convert indices back to actual class labels
        predictions = self.classes_[predicted_class_indices]
        
        return predictions

print("GaussianNaiveBayesFromScratch class implemented successfully!")

In [ ]:
# Step 6: Test the Gaussian PDF function
def test_gaussian_pdf():
    """
    Test our Gaussian PDF implementation.
    """
    # Create a simple test
    model = GaussianNaiveBayesFromScratch()
    
    # Test with standard normal distribution (mean=0, variance=1)
    test_values = np.array([-2, -1, 0, 1, 2])
    probabilities = [model._gaussian_pdf(x, mean=0, variance=1) for x in test_values]
    
    print("Testing Gaussian PDF with standard normal (μ=0, σ²=1):")
    for x, prob in zip(test_values, probabilities):
        print(f"P(x={x:2d} | μ=0, σ²=1) = {prob:.6f}")
    
    # Visualize the PDF
    x_range = np.linspace(-4, 4, 100)
    y_values = [model._gaussian_pdf(x, mean=0, variance=1) for x in x_range]
    
    plt.figure(figsize=(10, 6))
    plt.plot(x_range, y_values, 'b-', linewidth=2, label='Gaussian PDF (μ=0, σ²=1)')
    plt.scatter(test_values, probabilities, color='red', s=50, zorder=5, label='Test points')
    plt.xlabel('x')
    plt.ylabel('Probability Density')
    plt.title('Gaussian Probability Density Function')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

test_gaussian_pdf()

In [ ]:
# Step 7: Train Your Model
model = GaussianNaiveBayesFromScratch()
model.fit(X_train, y_train)

print("=" * 50)
print("LEARNED PARAMETERS")
print("=" * 50)

print(f"\nClasses found: {model.classes_}")
print(f"Class names: {[class_names[i] for i in model.classes_]}")

print("\nClass Priors P(C_k):")
for i, prior in enumerate(model.class_priors_):
    print(f"  P({class_names[i]}) = {prior:.4f}")

print("\nFeature Means by Class:")
for class_idx, class_name in enumerate(class_names):
    print(f"\n  {class_name}:")
    for feat_idx, feature_name in enumerate(feature_names):
        mean = model.feature_means_[class_idx, feat_idx]
        print(f"    μ({feature_name}) = {mean:.3f}")

print("\nFeature Variances by Class:")
for class_idx, class_name in enumerate(class_names):
    print(f"\n  {class_name}:")
    for feat_idx, feature_name in enumerate(feature_names):
        variance = model.feature_vars_[class_idx, feat_idx]
        std_dev = np.sqrt(variance)
        print(f"    σ²({feature_name}) = {variance:.3f}, σ = {std_dev:.3f}")

In [ ]:
# Step 8: Test Your Model
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("=" * 50)
print("MODEL PERFORMANCE")
print("=" * 50)
print(f"\nAccuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Show classification report
print("\nDetailed Classification Report:")
target_names = [class_names[i] for i in np.unique(y_test)]
print(classification_report(y_test, y_pred, target_names=target_names))

# Show confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix - Our Implementation')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Show some individual predictions
print("\nSample Predictions:")
print("-" * 80)
for i in range(min(10, len(X_test))):
    true_class = class_names[y_test[i]]
    pred_class = class_names[y_pred[i]]
    confidence = np.max(y_proba[i])
    
    print(f"Sample {i+1}: True={true_class:12} | Pred={pred_class:12} | Confidence={confidence:.3f}")
    prob_dict = {class_names[j]: y_proba[i][j] for j in range(len(class_names))}
    print(f"          Probabilities: {prob_dict}")
    print()

In [ ]:
# Step 9: Compare with Scikit-Learn
sklearn_model = GaussianNB()
sklearn_model.fit(X_train, y_train)

# Make predictions with sklearn model
sklearn_pred = sklearn_model.predict(X_test)
sklearn_proba = sklearn_model.predict_proba(X_test)

# Calculate sklearn accuracy
sklearn_accuracy = accuracy_score(y_test, sklearn_pred)

print("=" * 60)
print("COMPARISON: OUR IMPLEMENTATION vs SCIKIT-LEARN")
print("=" * 60)
print(f"\nOur Implementation:     {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Scikit-Learn:           {sklearn_accuracy:.4f} ({sklearn_accuracy*100:.2f}%)")
print(f"Difference:             {abs(accuracy - sklearn_accuracy):.6f}")

# Compare individual predictions
agreement = np.mean(y_pred == sklearn_pred)
print(f"\nPrediction Agreement:   {agreement:.4f} ({agreement*100:.2f}%)")

# Show parameter comparison
print("\nParameter Comparison:")
print("\nClass Priors:")
print(f"Our implementation: {model.class_priors_}")
print(f"Scikit-learn:       {sklearn_model.class_prior_}")

print("\nFeature Means (first few):")
print(f"Our implementation:\n{model.feature_means_[:, :2]}")
print(f"Scikit-learn:\n{sklearn_model.theta_[:, :2]}")

print("\nFeature Variances (first few):")
print(f"Our implementation:\n{model.feature_vars_[:, :2]}")
print(f"Scikit-learn:\n{sklearn_model.var_[:, :2]}")

# Show side-by-side predictions for disagreements
disagreements = (y_pred != sklearn_pred)
if np.any(disagreements):
    print(f"\nFound {np.sum(disagreements)} disagreements:")
    disagreement_indices = np.where(disagreements)[0]
    
    for idx in disagreement_indices[:5]:  # Show first 5 disagreements
        print(f"\nSample {idx}:")
        print(f"  True class:     {class_names[y_test[idx]]}")
        print(f"  Our prediction: {class_names[y_pred[idx]]} (conf: {np.max(y_proba[idx]):.3f})")
        print(f"  Sklearn pred:   {class_names[sklearn_pred[idx]]} (conf: {np.max(sklearn_proba[idx]):.3f})")
else:
    print("\nPerfect agreement! All predictions match.")

In [ ]:
# Step 10: Visualize Decision Boundaries
def visualize_decision_boundaries():
    """
    Visualize decision boundaries and feature distributions.
    """
    
    # Choose two features for 2D visualization (petal length and width)
    feature_indices = [2, 3]  # Petal length and petal width
    feature_x, feature_y = feature_indices
    
    X_2d = X_train[:, feature_indices]
    X_test_2d = X_test[:, feature_indices]
    
    # Train a 2D version of our model
    model_2d = GaussianNaiveBayesFromScratch()
    model_2d.fit(X_2d, y_train)
    
    # Create a mesh for decision boundary visualization
    h = 0.02  # Step size
    x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
    y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict probabilities for the entire mesh
    mesh_points = np.c_[xx.ravel(), yy.ravel()]
    Z = model_2d.predict_proba(mesh_points)
    Z_class = np.argmax(Z, axis=1)
    Z_class = Z_class.reshape(xx.shape)
    
    # Create the visualization
    plt.figure(figsize=(15, 10))
    
    # Plot decision boundaries
    plt.subplot(2, 2, 1)
    plt.contourf(xx, yy, Z_class, alpha=0.4, cmap='viridis')
    
    # Plot training data
    colors = ['red', 'green', 'blue']
    for i, class_name in enumerate(class_names):
        mask = (y_train == i)
        plt.scatter(X_2d[mask, 0], X_2d[mask, 1], 
                   c=colors[i], label=class_name, alpha=0.8)
    
    plt.xlabel(feature_names[feature_x])
    plt.ylabel(feature_names[feature_y])
    plt.title('Decision Boundaries (Training Data)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot test predictions
    plt.subplot(2, 2, 2)
    plt.contourf(xx, yy, Z_class, alpha=0.4, cmap='viridis')
    
    y_test_2d = model_2d.predict(X_test_2d)
    
    # Plot correctly classified test points
    correct = (y_test == y_test_2d)
    for i, class_name in enumerate(class_names):
        mask = (y_test == i) & correct
        plt.scatter(X_test_2d[mask, 0], X_test_2d[mask, 1], 
                   c=colors[i], label=f'{class_name} (correct)', marker='o', s=50)
    
    # Plot misclassified test points
    incorrect = ~correct
    if np.any(incorrect):
        plt.scatter(X_test_2d[incorrect, 0], X_test_2d[incorrect, 1], 
                   c='black', marker='x', s=100, label='Misclassified')
    
    plt.xlabel(feature_names[feature_x])
    plt.ylabel(feature_names[feature_y])
    plt.title('Test Predictions')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot Gaussian contours
    plt.subplot(2, 2, 3)
    
    for class_idx, (class_name, color) in enumerate(zip(class_names, colors)):
        # Get mean and variance for this class and these two features
        mean_x = model_2d.feature_means_[class_idx, 0]
        mean_y = model_2d.feature_means_[class_idx, 1]
        var_x = model_2d.feature_vars_[class_idx, 0]
        var_y = model_2d.feature_vars_[class_idx, 1]
        
        # Create ellipse for 1 and 2 standard deviations
        from matplotlib.patches import Ellipse
        
        for std_mult in [1, 2]:
            ellipse = Ellipse((mean_x, mean_y), 
                             2 * std_mult * np.sqrt(var_x), 
                             2 * std_mult * np.sqrt(var_y),
                             alpha=0.3 / std_mult, 
                             color=color)
            plt.gca().add_patch(ellipse)
        
        # Plot class center
        plt.scatter(mean_x, mean_y, c=color, s=100, marker='*', 
                   edgecolor='black', label=f'{class_name} center')
    
    # Plot all training data
    for i, class_name in enumerate(class_names):
        mask = (y_train == i)
        plt.scatter(X_2d[mask, 0], X_2d[mask, 1], 
                   c=colors[i], alpha=0.6, s=20)
    
    plt.xlabel(feature_names[feature_x])
    plt.ylabel(feature_names[feature_y])
    plt.title('Gaussian Distributions (1σ and 2σ contours)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Feature importance heatmap
    plt.subplot(2, 2, 4)
    
    # Calculate feature "importance" as variance of means across classes
    feature_importance = np.var(model.feature_means_, axis=0)
    
    plt.bar(range(len(feature_names)), feature_importance, color='skyblue')
    plt.xlabel('Features')
    plt.ylabel('Variance of Class Means')
    plt.title('Feature Discrimination Power')
    plt.xticks(range(len(feature_names)), 
              [name.split()[0] for name in feature_names], rotation=45)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_decision_boundaries()

In [ ]:
# Step 11: Cross-Validation Extension
def cross_validate_model(X, y, k=5):
    """
    Perform k-fold cross-validation on your Gaussian Naive Bayes model.
    
    Parameters:
    X : array-like, features
    y : array-like, labels  
    k : int, number of folds
    
    Returns:
    scores : list of accuracy scores for each fold
    """
    
    kfold = KFold(n_splits=k, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kfold.split(X):
        X_fold_train, X_fold_val = X[train_idx], X[val_idx]
        y_fold_train, y_fold_val = y[train_idx], y[val_idx]
        
        # Train model on fold
        fold_model = GaussianNaiveBayesFromScratch()
        fold_model.fit(X_fold_train, y_fold_train)
        
        # Predict and score
        y_fold_pred = fold_model.predict(X_fold_val)
        fold_score = accuracy_score(y_fold_val, y_fold_pred)
        scores.append(fold_score)
    
    return scores

# Run cross-validation
cv_scores = cross_validate_model(X, y, k=5)
print(f"Cross-validation scores: {[f'{score:.4f}' for score in cv_scores]}")
print(f"Mean CV accuracy: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

# Compare with sklearn cross-validation
from sklearn.model_selection import cross_val_score
sklearn_cv_scores = cross_val_score(GaussianNB(), X, y, cv=5)
print(f"Sklearn CV scores: {[f'{score:.4f}' for score in sklearn_cv_scores]}")
print(f"Sklearn mean CV accuracy: {np.mean(sklearn_cv_scores):.4f} ± {np.std(sklearn_cv_scores):.4f}")

In [ ]:
# Step 12: Summary and Analysis
print("=" * 60)
print("FINAL ANALYSIS AND SUMMARY")
print("=" * 60)

print("\n1. IMPLEMENTATION SUCCESS:")
print(f"   ✓ Gaussian Naive Bayes implemented from scratch")
print(f"   ✓ Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   ✓ Agreement with sklearn: {agreement:.4f} ({agreement*100:.2f}%)")
print(f"   ✓ Cross-validation: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

print("\n2. KEY INSIGHTS:")
# Find most discriminative feature
feature_importance = np.var(model.feature_means_, axis=0)
most_important_idx = np.argmax(feature_importance)
print(f"   • Most discriminative feature: {feature_names[most_important_idx]}")
print(f"   • Feature discrimination scores: {feature_importance.round(3)}")

# Check which classes are most confused
cm = confusion_matrix(y_test, y_pred)
print(f"   • Classes most often confused: Check confusion matrix above")

print("\n3. MODEL CHARACTERISTICS:")
print(f"   • Assumes feature independence (naive assumption)")
print(f"   • Models each feature with Gaussian distribution")
print(f"   • Uses Bayes' theorem for classification")
print(f"   • Handles numerical stability with log probabilities")

print("\n4. WHEN TO USE GAUSSIAN NAIVE BAYES:")
print(f"   • Small to medium datasets")
print(f"   • Features are approximately normally distributed")
print(f"   • Need fast, interpretable classifier")
print(f"   • Baseline model for comparison")
print(f"   • When features are reasonably independent")

print("\n5. IMPLEMENTATION LEARNING:")
print(f"   • Mathematical foundation: Bayes' theorem + Gaussian PDF")
print(f"   • Numerical considerations: log probabilities, variance smoothing")
print(f"   • Parameter estimation: sample mean and variance")
print(f"   • Prediction: argmax of posterior probabilities")

print("\n" + "="*60)
print("CONGRATULATIONS! You have successfully implemented")
print("Gaussian Naive Bayes from scratch with complete")
print("mathematical understanding and practical skills!")
print("="*60)